# Concept Notes — Proximal Gradient, Subgradient & Acceleration
**MIT 6.7220 | Nikhilesh Belulkar**

---

## 1. The Problem Structure

We want to solve:
$$\min_x F(x) = \underbrace{\frac{1}{2}\|Ax - b\|_2^2}_{f(x)\ \text{smooth}} + \underbrace{\lambda\|x\|_1}_{r(x)\ \text{non-smooth}}$$

The key insight is to **split** $F$ into two parts and treat each differently:

| Part | Function | Property | How we handle it |
|------|----------|----------|------------------|
| $f(x)$ | $\frac{1}{2}\|Ax-b\|_2^2$ | Smooth, differentiable | Gradient step |
| $r(x)$ | $\lambda\|x\|_1$ | Non-smooth | Proximal operator |

## 2. Why Can't We Just Use Gradient Descent?

Gradient descent requires $F$ to be differentiable everywhere. But $\|x\|_1 = \sum_i |x_i|$ is **not differentiable at $x_i = 0$** — it has a kink.

So we need smarter methods that can handle the non-smooth $\ell_1$ term.

## 3. The Proximal Operator

The proximal operator of a function $r$ with step size $h$ is defined as:
$$\text{prox}_{hr}(v) = \arg\min_y \left\{ r(y) + \frac{1}{2h}\|y - v\|_2^2 \right\}$$

**Intuition:** Find the point $y$ that:
- has small $r(y)$ (minimizes the non-smooth part), and
- stays close to $v$ (the $\frac{1}{2h}\|y-v\|^2$ term is a proximity penalty)

**Important:** The proximal operator gives the **exact** minimizer of this subproblem — it does NOT approximate $r$.

### For $r(x) = \lambda\|x\|_1$: Soft Thresholding

$$\text{prox}_{h\lambda\|\cdot\|_1}(v)_i = \mathcal{S}_{h\lambda}(v_i) = \begin{cases} v_i - h\lambda & \text{if } v_i > h\lambda \\ 0 & \text{if } |v_i| \leq h\lambda \\ v_i + h\lambda & \text{if } v_i < -h\lambda \end{cases}$$

This shrinks small values to zero and shifts large values toward zero — promoting sparsity.

## 4. Proximal Gradient Method

### Step 1: Where does $z_t$ come from?

Start with the subproblem we want to solve at each iteration. We linearize $f$ around $x_t$ and add a quadratic trust-region:
$$\min_y \left\{ \underbrace{\langle \nabla f(x_t),\, y - x_t \rangle}_{\text{linearized } f} + \underbrace{\frac{1}{2h}\|y - x_t\|^2}_{\text{trust region}} + r(y) \right\}$$

Focus just on the first two terms (ignore $r(y)$ for now). Let $g = \nabla f(x_t)$ and $c = x_t$ to keep it clean:
$$\langle g,\, y - c \rangle + \frac{1}{2h}\|y - c\|^2$$

**Complete the square** — pull the $\frac{1}{2h}$ out and write everything as a single squared norm:
$$= \frac{1}{2h}\left[ \|y - c\|^2 + 2h\langle g,\, y - c \rangle \right]$$
$$= \frac{1}{2h}\left[ \|y - c\|^2 + 2h\langle g,\, y - c \rangle + h^2\|g\|^2 \right] - \frac{h}{2}\|g\|^2$$
$$= \frac{1}{2h}\|y - (c - hg)\|^2 - \frac{h}{2}\|g\|^2$$

The last term $-\frac{h}{2}\|g\|^2$ is a constant (no $y$ in it), so it does not affect the minimizer. Define:
$$z_t := c - hg = x_t - h\nabla f(x_t)$$

This is just the **standard gradient step** on $f$. The full subproblem simplifies to:
$$x_{t+1} = \arg\min_y \left\{ \frac{1}{2h}\|y - z_t\|^2 + r(y) \right\} = \text{prox}_{hr}(z_t)$$

**So $z_t$ is not mysterious — it is just the gradient descent step on the smooth part $f$. The completing-the-square algebra shows that taking a gradient step on $f$ and then solving the proximal subproblem are equivalent.**

---

### Step 2: Where does the optimality condition come from?

We need to solve:
$$\min_y\; \phi(y) := r(y) + \frac{1}{2h}\|y - z_t\|^2$$

For any convex function $\phi$, a point $y^*$ is a minimizer **if and only if** zero is in its subdifferential:
$$y^* \text{ minimizes } \phi \iff 0 \in \partial \phi(y^*)$$

This is the convex analogue of "set the derivative to zero." Now compute $\partial \phi$:
$$\partial \phi(y) = \partial r(y) + \nabla\!\left[\frac{1}{2h}\|y - z_t\|^2\right]$$

The second term is smooth so we can just take its regular gradient:
$$\nabla\!\left[\frac{1}{2h}\|y - z_t\|^2\right] = \frac{1}{h}(y - z_t)$$

So the optimality condition $0 \in \partial\phi(y^*)$ becomes:
$$0 \in \partial r(y^*) + \frac{1}{h}(y^* - z_t)$$

**For $r = \lambda\|x\|_1$:** since $\|x\|_1 = \sum_i |x_i|$ is separable, the subdifferential splits component-wise:
$$0 \in \lambda\,\partial|y_i^*| + \frac{1}{h}(y_i^* - (z_t)_i) \qquad \text{for each } i$$

Each of these is a scalar problem. Solving case-by-case (exactly as in Q1) gives the **soft-thresholding** solution:
$$y_i^* = \mathcal{S}_{h\lambda}((z_t)_i)$$

**Summary of what's happening:**

$$\underbrace{z_t = x_t - h\nabla f(x_t)}_{\text{gradient step: "where would we go ignoring } r\text{?"}} \xrightarrow{\text{prox}} \underbrace{x_{t+1} = \mathcal{S}_{h\lambda}(z_t)}_{\text{pull } z_t \text{ back toward sparsity}}$$

The gradient step moves in the direction that reduces $f$. The proximal step then enforces the $\ell_1$ penalty by shrinking small components to zero — exactly solving the non-smooth part rather than approximating it.

## 5. Subgradient Method

A subgradient $g \in \partial F(x)$ generalizes the gradient to non-smooth functions. For our $F$:
$$g_t = \underbrace{A^\top(Ax_t - b)}_{\nabla f(x_t)} + \underbrace{\lambda\,\text{sign}(x_t)}_{\in\, \lambda\,\partial\|x_t\|_1}$$

The subgradient iteration is:
$$x_{t+1} = x_t - h_t\, g_t$$

### Polyak Step Size
Instead of a fixed $h$, Polyak's rule adapts the step size:
$$h_t = \frac{F(x_t) - F^*}{\|g_t\|^2}$$

### Key difference from proximal gradient:

| | Proximal Gradient | Subgradient |
|---|---|---|
| Handles $r$ | Exactly (via prox) | Approximately (via subgradient) |
| Descent guarantee | Yes, every step | No — can increase |
| Convergence rate | $O(1/t)$ | $O(1/\sqrt{t})$ |
| Step size | Fixed $h = 1/L_f$ | Adaptive (Polyak) |

## 6. Nesterov Acceleration (FISTA)

### Why accelerate?

Proximal gradient converges at $O(1/t)$. Nesterov's trick improves this to $O(1/t^2)$ — **for free**, with almost no extra computation.

### The idea: look ahead before you step

Instead of taking the gradient step from $x_t$, maintain a **momentum point** $y_t$ (an extrapolation ahead of $x_t$) and step from there:

$$x_{t+1} = \text{prox}_{hr}\!\left(y_t - h\nabla f(y_t)\right)$$

$$\alpha_{t+1} = \frac{1 + \sqrt{1 + 4\alpha_t^2}}{2}$$

$$y_{t+1} = x_{t+1} + \frac{\alpha_t - 1}{\alpha_{t+1}}(x_{t+1} - x_t)$$

The $y_{t+1}$ update extrapolates in the direction of past progress — like "overshooting" to pre-correct for the zig-zagging that normal gradient descent does.

### Convergence rates summary:

| Method | Rate | 
|--------|------|
| Subgradient | $O(1/\sqrt{t})$ |
| Proximal Gradient | $O(1/t)$ |
| FISTA (Accelerated) | $O(1/t^2)$ ← optimal for first-order methods |

## 7. Lipschitz Constant and Step Size

For the step size $h = 1/L_f$ to guarantee convergence, $L_f$ must be the **Lipschitz constant of $\nabla f$** — i.e., how fast the gradient can change:
$$\|\nabla f(x) - \nabla f(y)\|_2 \leq L_f \|x - y\|_2 \quad \forall\, x, y$$

For $f(x) = \frac{1}{2}\|Ax-b\|_2^2$:
- Hessian: $\nabla^2 f(x) = A^\top A$ (constant)
- $L_f = \lambda_{\max}(A^\top A)$ — the largest eigenvalue

**Why?** The Hessian tells you the curvature of $f$. The largest eigenvalue is the worst-case curvature. The step size $h = 1/L_f$ ensures you don't overshoot.

---

## 8. Problem 2 Q2: Newton Convergence Rate — How to Think About It

**Goal:** Show that if $\|x_t - x^*\|_{x_t} < 1/2$, then one Newton step satisfies:
$$\|x_{t+1} - x^*\|_{x_t} \leq \frac{\|x_t - x^*\|^2_{x_t}}{1 - \|x_t - x^*\|_{x_t}}$$

Both sides use the **same** $x_t$ norm.

---

### The cubic remainder bound — what it says

The pset states: for $q_x(y) = f(x) + \langle\nabla f(x), y-x\rangle + \frac{1}{2}\|y-x\|_x^2$ and $r = \|y-x\|_x < 1$:

$$|f(y) - q_x(y)| \leq \frac{r^3}{3(1-r)}$$

This bounds how much the **function value** $f(y)$ deviates from the quadratic Taylor model $q_x(y)$ centered at $x$.

**For Newton convergence we need a gradient-level version of this.** The gradient of $q_x(y)$ w.r.t. $y$ is $\nabla f(x) + \nabla^2 f(x)(y-x)$. The question is: how far is $\nabla f(y)$ from this linear approximation? The answer (which can be derived from self-concordance using the integral representation of the gradient, much like the function-value bound is derived by integration from the self-concordance condition):

$$\boxed{\|\nabla f(y) - \nabla f(x) - \nabla^2 f(x)(y-x)\|_{x,*} \leq \frac{r^2}{1-r}}$$

**How the two are related:** the function-value bound ($r^3/3(1-r)$) is essentially the integral of the gradient bound ($r^2/(1-r)$) from $0$ to $r$. Differentiating the cubic bound "morally" gives the gradient bound — and the $r^3/3(1-r)$ becomes $r^2/(1-r)$ after differentiation. So the $r^3$ and $3$ do not appear in the Newton step proof; they live in the function-value statement, while the Newton step proof uses $r^2/(1-r)$.

---

### Step 1: Write out $x_{t+1}$ explicitly

The Newton step is $n(x_t) = -[\nabla^2 f(x_t)]^{-1}\nabla f(x_t)$, giving:
$$x_{t+1} = x_t + n(x_t) = x_t - [\nabla^2 f(x_t)]^{-1}\nabla f(x_t)$$

Subtract $x^*$ from both sides:
$$x_{t+1} - x^* = (x_t - x^*) - [\nabla^2 f(x_t)]^{-1}\nabla f(x_t)$$

---

### Step 2: Try the triangle inequality — and see exactly why it fails

Apply triangle inequality to $x_{t+1} - x^* = (x_t - x^*) + n(x_t)$:

$$\|x_{t+1} - x^*\|_{x_t} \leq \underbrace{\|x_t - x^*\|_{x_t}}_{e_t} + \underbrace{\|n(x_t)\|_{x_t}}_{\lambda(x_t)\ \text{(Newton decrement)}}$$

This gives $\|x_{t+1} - x^*\|_{x_t} \leq e_t + \lambda(x_t)$, which is **strictly larger than $e_t$**. No contraction — useless.

**Why it fails:** Triangle inequality bounds $\|a + b\|$ by $\|a\| + \|b\|$, which is tight when $a$ and $b$ point in the same direction. Near a minimizer, Newton's method works precisely because $n(x_t) \approx -(x_t - x^*)$: the Newton step is designed to almost exactly cancel the error $(x_t - x^*)$. Triangle inequality throws away this cancellation entirely.

---

### Step 3: Exploit the cancellation — use $\nabla f(x^*) = 0$

Since $x^*$ minimizes $f$, we have $\nabla f(x^*) = 0$. So $\nabla f(x_t) = \nabla f(x_t) - \nabla f(x^*)$. Substitute:

$$x_{t+1} - x^* = (x_t - x^*) - [\nabla^2 f(x_t)]^{-1}\underbrace{[\nabla f(x_t) - \nabla f(x^*)]}_{\text{write as linear part + remainder}}$$

Now split the gradient difference using the linear (Hessian) term and the nonlinear remainder:
$$\nabla f(x_t) - \nabla f(x^*) = \underbrace{\nabla^2 f(x_t)(x_t - x^*)}_{\text{linear part}} + \underbrace{[\nabla f(x_t) - \nabla f(x^*) - \nabla^2 f(x_t)(x_t - x^*)]}_{\text{nonlinear remainder }\rho}$$

Substitute back:
$$x_{t+1} - x^* = (x_t - x^*) - [\nabla^2 f(x_t)]^{-1}[\nabla^2 f(x_t)(x_t - x^*) + \rho]$$

$$= (x_t - x^*) - (x_t - x^*) - [\nabla^2 f(x_t)]^{-1}\rho = -[\nabla^2 f(x_t)]^{-1}\rho$$

**The $(x_t - x^*)$ terms cancel exactly.** The entire Newton step error equals $-[\nabla^2 f(x_t)]^{-1}\rho$, where $\rho$ is the nonlinear (non-quadratic) part of the gradient — exactly what makes $f$ deviate from a quadratic. If $f$ were quadratic, $\rho = 0$ and Newton would converge in one step.

---

### Step 4: Bound the remainder using the gradient cubic bound

Taking the $x_t$-norm of $x_{t+1} - x^* = -[\nabla^2 f(x_t)]^{-1}\rho$:

$$\|x_{t+1} - x^*\|_{x_t} = \|[\nabla^2 f(x_t)]^{-1}\rho\|_{x_t} = \|\rho\|_{x_t, *}$$

(the last equality holds because $\|H^{-1}g\|_H = \|g\|_{H^{-1}} = \|g\|_{x_t,*}$ by definition of the dual local norm.)

Now apply the gradient-level cubic bound with $x = x_t$, $y = x^*$, $r = \|x^* - x_t\|_{x_t} = e_t$:

$$\|\rho\|_{x_t,*} = \|\nabla f(x_t) - \nabla f(x^*) - \nabla^2 f(x_t)(x_t - x^*)\|_{x_t,*} \leq \frac{e_t^2}{1 - e_t}$$

Therefore:
$$\boxed{\|x_{t+1} - x^*\|_{x_t} \leq \frac{e_t^2}{1-e_t} = \frac{\|x_t - x^*\|_{x_t}^2}{1 - \|x_t - x^*\|_{x_t}}}$$

When $e_t < 1/2$: the right-hand side is at most $2e_t^2 \ll e_t$ — the error shrinks quadratically.

---

### Step 5: Why does this NOT immediately imply quadratic convergence?

The bound gives $\|x_{t+1} - x^*\|_{x_t} \leq e_t^2/(1-e_t)$, which looks quadratic. But **quadratic convergence** requires:
$$\|x_{t+1} - x^*\|_{x_{t+1}} \leq C \cdot e_t^2$$

We have the new error in the **old norm** $\|\cdot\|_{x_t}$; quadratic convergence needs it in the **new norm** $\|\cdot\|_{x_{t+1}}$.

To convert, the local-norm change theorem (from the pset) says for $r = \|y-x\|_x < 1$:
$$\|h\|_y \leq \frac{1}{1-r}\|h\|_x$$

Applying with $y = x_{t+1}$, $x = x_t$, $r = \|x_{t+1} - x_t\|_{x_t} = \lambda(x_t)$:
$$\|x_{t+1} - x^*\|_{x_{t+1}} \leq \frac{1}{1-\lambda(x_t)}\|x_{t+1} - x^*\|_{x_t} \leq \frac{e_t^2}{(1-e_t)(1-\lambda(x_t))}$$

This still has $\lambda(x_t)$ (the Newton decrement) in it, which is not directly controlled by $e_t$ without further work. Bonus Q3 closes this gap: it shows $\lambda(x_{t+1}) \leq \left(\frac{\lambda(x_t)}{1-\lambda(x_t)}\right)^2$, establishing quadratic convergence of the Newton decrement.

---

### Summary

| Approach | Bound obtained | Verdict |
|----------|---------------|---------|
| Triangle: $\|(x_t-x^*) + n(x_t)\|_{x_t} \leq e_t + \lambda(x_t)$ | larger than $e_t$ | **Useless** — no contraction |
| Cancellation + gradient cubic: $\|[\nabla^2 f]^{-1}\rho\|_{x_t} \leq r^2/(1-r)$ | $e_t^2/(1-e_t)$ | **Quadratic shrinkage** |

The function-value cubic bound $r^3/3(1-r)$ from the pset is the integrated form; the gradient-level bound $r^2/(1-r)$ is what drives the Newton step error, and can be thought of as the "derivative" of $r^3/3(1-r)$.

---

## 9. Where Do the Cubic Bounds Come From? (Derivation Chain)

The two cubic bounds are related, but **not** by differentiating one to get the other — the function-value bound is derived by **integrating** the gradient bound, which is itself derived from self-concordance. The chain goes:

$$\underbrace{|D^3f[h,h,h]| \leq 2\|h\|_x^3}_{\text{self-concordance}} \;\Longrightarrow\; \underbrace{\|\nabla f(y) - \nabla f(x) - \nabla^2 f(x)(y-x)\|_{x,*} \leq \frac{r^2}{1-r}}_{\text{gradient bound}} \;\Longrightarrow\; \underbrace{|f(y) - q_x(y)| \leq \frac{r^3}{3(1-r)}}_{\text{function-value bound}}$$

---

### Step A: Get the gradient bound from self-concordance

By polarization of the self-concordance condition $|D^3f(x)[h,h,h]| \leq 2\|h\|_x^3$, for any three directions $u,v,w$:
$$|D^3f(x)[u,v,w]| \leq 2\|u\|_x\|v\|_x\|w\|_x$$

Now fix $x$, $y$, and any test direction $v$. The gradient remainder in direction $v$ is:
$$\langle \nabla f(y) - \nabla f(x) - \nabla^2 f(x)(y-x),\, v\rangle$$

Use the fundamental theorem of calculus twice. First, the gradient difference:
$$\nabla f(y) - \nabla f(x) = \int_0^1 \nabla^2 f(x + t(y-x))(y-x)\, dt$$

So the remainder in direction $v$ is:
$$\int_0^1 \underbrace{\langle [\nabla^2 f(x+t(y-x)) - \nabla^2 f(x)](y-x),\, v\rangle}_{= \int_0^t D^3f(x+s(y-x))[y-x,\, y-x,\, v]\, ds} dt$$

Switching the order of integration:
$$= \int_0^1 \int_0^t D^3f(x+s(y-x))[y-x, y-x, v]\, ds\, dt$$

Apply the polarized bound and Hessian stability (at the point $x + s(y-x)$, which is at $x_t$-distance $sr$ from $x$, so norms scale by $1/(1-sr)$):
$$|D^3f(x+s(y-x))[y-x, y-x, v]| \leq \frac{2r^2\|v\|_x}{(1-sr)^3}$$

where $r = \|y-x\|_x$. Integrating:
$$|\langle \text{remainder}, v\rangle| \leq 2r^2\|v\|_x \int_0^1 \int_0^t \frac{ds\, dt}{(1-sr)^3}$$

The double integral evaluates to $\frac{1}{2(1-r)}$ (compute by substituting $u=1-sr$ and integrating in $s$ then $t$):

$$= 2r^2\|v\|_x \cdot \frac{1}{2(1-r)} = \frac{r^2\|v\|_x}{1-r}$$

Taking the supremum over $\|v\|_x = 1$ gives the **gradient bound**:
$$\|\nabla f(y) - \nabla f(x) - \nabla^2 f(x)(y-x)\|_{x,*} \leq \frac{r^2}{1-r}$$

---

### Step B: Integrate the gradient bound to get the function-value bound

Now compute $f(y) - q_x(y)$ using the fundamental theorem of calculus:
$$f(y) - q_x(y) = \int_0^1 \langle \underbrace{\nabla f(x+t(y-x)) - \nabla f(x) - t\nabla^2 f(x)(y-x)}_{\text{gradient remainder at } x+t(y-x)},\; y-x\rangle\, dt$$

Apply the gradient bound to the gradient remainder at $x + t(y-x)$ (which is at distance $tr$ from $x$):
$$\|\text{gradient remainder at } x+t(y-x)\|_{x,*} \leq \frac{(tr)^2}{1-tr}$$

So:
$$|f(y) - q_x(y)| \leq \int_0^1 \frac{t^2 r^2}{1-tr} \cdot \|y-x\|_x\, dt = r^3 \int_0^1 \frac{t^2}{1-tr}\, dt$$

Now bound the integral using $1-tr \geq 1-r$ for $t \in [0,1]$:
$$\int_0^1 \frac{t^2}{1-tr}\, dt \leq \frac{1}{1-r}\int_0^1 t^2\, dt = \frac{1}{3(1-r)}$$

Therefore:
$$|f(y) - q_x(y)| \leq \frac{r^3}{3(1-r)}$$

---

### The key takeaway

The function-value cubic bound $r^3/3(1-r)$ is **downstream** of the gradient bound $r^2/(1-r)$: it comes from integrating the gradient bound one more time. So for the Newton step convergence proof, we use the gradient bound directly (one integration level earlier in the chain), and the $r^3/3(1-r)$ never needs to appear.